# Swin temporal stable OOF diagnostics

Creates validation OOF diagnostics from the 5 fold Swin temporal stable checkpoints. No retraining.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'torch:', torch.__version__)

from run_oof_diagnostics import run

DATA_ROOT = Path('/kaggle/input/solafune-dataset-v2')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/datasets/suioshion/solafune-dataset-v2')

MODEL_ROOT = Path('/kaggle/input/swin-temporal-stable-models')
if not MODEL_ROOT.exists():
    MODEL_ROOT = Path('/kaggle/input/datasets/suioshion/swin-temporal-stable-models')

print('DATA_ROOT:', DATA_ROOT)
print('MODEL_ROOT:', MODEL_ROOT)


class Args:
    root = '/kaggle/working'
    kaggle_input_root = str(DATA_ROOT)
    checkpoint_root = str(MODEL_ROOT)
    folds = '0,1,2,3,4'
    batch_size = 16
    workers = 2
    model_subdir = 'swin_v2_temporal_stable'
    output_dir = 'outputs/oof_swin_v2_temporal_stable'
    use_temporal_differences = True
    use_temporal_summary = True
    use_location_features = False
    location_metadata_path = None
    no_amp = True


output_dir = run(Args())

import pandas as pd
for name in ['oof_overall_summary.csv', 'oof_fold_summary.csv', 'oof_location_summary.csv', 'oof_month_summary.csv', 'oof_satellite_month_summary.csv']:
    print('\n', name)
    display(pd.read_csv(Path(output_dir) / name).head(30))
